In [ ]:
Sys.time()

In [ ]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(Seurat))

library(tidyverse)

In [ ]:
addArchRGenome("hg38")
addArchRThreads(threads = 50)

In [ ]:
projdir <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples <- read.table(paste0(projdir, 'txt/samples.txt'), sep='\t')$V1

In [ ]:
s = samples[1]
paste0(projdir, '/rds/rna/for_archr/', s, '.rds')

In [ ]:
n <- length(samples)

s = samples[1]

rds_path <- paste0(projdir, '/rds/rna/for_archr/', s, '.rds')
seur_obj <- readRDS(rds_path)

for (i in seq_along(samples[2:n])) {
    s = samples[i]
    print(s)
    
    rds_path <- paste0(projdir, '/rds/rna/for_archr/', s, '.rds')
    tmp_obj <- readRDS(rds_path)
    seur_obj <- cbind(seur_obj, tmp_obj)
}

In [ ]:
seur_obj

In [ ]:
cbind(seur_obj, seur_obj)

In [ ]:
tmp_obj

In [ ]:
seur_names <- rownames(seur_obj)
tmp_names <- rownames(tmp_obj)
sum(seur_names == tmp_names)

In [ ]:
mask <- seur_names %in% tmp_names
sum(mask)

In [ ]:
dim(seur_obj)
dim(tmp_obj)

In [ ]:
test_obj <- tmp_obj[seur_names, ]

In [ ]:
test_names <- rownames(test_obj)
sum(seur_names == test_names)

In [ ]:
one <- rownames(assays(seur_obj)$counts)
two <- rownames(assays(test_obj)$counts)
sum(one == two)

In [ ]:
merged_obj <- cbind(seur_obj, test_obj)

In [ ]:
### finally...
rowRanges(test_obj) = rowRanges(seur_obj)

In [ ]:
merged_obj <- cbind(seur_obj, test_obj)

In [ ]:
merged_obj

### put it all together
so for some reason, all the genes are here but the order is wrong

In [ ]:
n <- length(samples) + 1

s = samples[1]

rds_path <- paste0(projdir, '/rds/rna/for_archr/', s, '.rds')
seur_obj <- readRDS(rds_path)
seur_names <- rownames(seur_obj)


for (i in seq_along(samples[2:n])) {
    s = samples[i]
    print(s)
    
    rds_path <- paste0(projdir, '/rds/rna/for_archr/', s, '.rds')
    tmp_obj <- readRDS(rds_path)
    
    tmp_obj <- tmp_obj[seur_names, ]
    rowRanges(tmp_obj) = rowRanges(seur_obj)
    
    seur_obj <- cbind(seur_obj, tmp_obj)
}

In [ ]:
dim(seur_obj)

In [ ]:
seur_obj

In [ ]:
out <- paste0(projdir, '/rds/rna/for_archr/merged.rds')
saveRDS(seur_obj, out)

### help me

In [ ]:
proj_path <- paste0(projdir, 'atac/archr/processed/YS3_filtered_qc')
proj <- loadArchRProject(path = proj_path, force = FALSE, showLogo = TRUE)

In [ ]:
seur_obj

In [ ]:
proj <- addGeneExpressionMatrix(
  input = proj,
  seRNA = seur_obj,
  chromSizes = getChromSizes(proj),
  excludeChr = c("chrM", "chrY"),
  scaleTo = 10000,
  verbose = TRUE,
  threads = getArchRThreads(),
  parallelParam = NULL,
  force = TRUE,
  logFile = createLogFile("addGeneExpressionMatrix")
)

In [ ]:
proj

In [ ]:
df <- as.data.frame(proj@cellColData)
dim(df)

In [ ]:
head(df)

In [ ]:
out_path <- paste0(projdir,'atac/archr/processed/YS3_atac_filter_with_rna')
proj <- saveArchRProject(ArchRProj = proj,
                         outputDirectory = out_path)

### subset ArchR to the 27K that pass QC in both?

In [ ]:
meta_path <- paste0(projdir, '/csv/joint/intersect_muon_meta.csv')
filtered_meta <- read.csv(meta_path, sep='\t', header=1)
dim(filtered_meta)

In [ ]:
head(filtered_meta)

In [ ]:
subset_bcs <- filtered_meta$archr_barcode
subset_proj <- subsetCells(
      ArchRProj = proj,
      cellNames = subset_bcs
)

In [ ]:
subset_proj

In [ ]:
out_path <- paste0(projdir, 'atac/archr/processed/YS3_intersect_with_rna')
subset_proj <- saveArchRProject(ArchRProj = subset_proj,
                         outputDirectory = out_path)